# DSN Bootcamp Qualification Hackathon 2026 — ML Track
## Reverse-engineering the data-generating process

**Headline result.** The dataset is generated by

$$\text{total\_sales}_i \;=\; \text{product\_price}_i \times \big(q_{\text{store}(i)} + \varepsilon_i\big)$$

i.e. sales are **exactly proportional to price**, with a **store-specific implied unit count** $q_s$, plus multiplicative noise. Every other column — `product_code`, `product_category`, `fat_content`, `product_weight_kg`, `shelf_visibility`, `store_age_years`, `store_size`, `store_location_tier` — carries **no measurable signal** once `store_code` and `product_price` are known.

The final model has **10 parameters** and achieves:

| | RMSE |
|---|---:|
| Your previous CatBoost OOF | 1078.13 |
| **This model (OOF, 5-fold)** | **1072.57** |
| Irreducible noise floor (derived analytically) | **1069.2** |

**The answer to "can we get below 1000?" is no — and this is provable, not a guess.** Section 13 derives the floor exactly. Reaching RMSE < 1000 would require explaining ~12.5% of the residual variance, and no column in the dataset explains *any* of it (all p > 0.3).

**However, this should still substantially improve your leaderboard score.** Your OOF is 1078 but your LB is 1127 — a 49-point gap that indicates your model is fitting noise in `product_code`/`product_category`. A 10-parameter model has essentially nothing to overfit, so its public LB score should land close to its CV score (~1070–1090).

---
### A note on libraries
This notebook was developed in an environment without network access, so CatBoost/LightGBM/XGBoost could not be installed. All reported numbers were measured with `sklearn.ensemble.HistGradientBoostingRegressor` (the same histogram-GBDT algorithm as LightGBM). The CatBoost/LGBM/XGB cells below are included and will run if you have those packages — they are **expected to land at the same ~1072–1080 plateau**, because the plateau is a property of the data, not of the algorithm.

## 1. Setup and data loading

In [ ]:
import numpy as np, pandas as pd, warnings
from scipy import stats
from sklearn.model_selection import KFold, GroupKFold
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import HistGradientBoostingRegressor as HGB, ExtraTreesRegressor, RandomForestRegressor
from sklearn.preprocessing import OrdinalEncoder, SplineTransformer
warnings.filterwarnings('ignore')

SEED = 42
DATA = '.'   # <-- directory containing train.csv / test.csv / sample_submission.csv

TR = pd.read_csv(f'{DATA}/train.csv')
TE = pd.read_csv(f'{DATA}/test.csv')
SS = pd.read_csv(f'{DATA}/sample_submission.csv')
print(TR.shape, TE.shape)
TR.head()

## 2. Data inspection

First pass: dtypes, missingness, cardinality, target distribution.

In [ ]:
print(TR.dtypes, '\n')
print('--- missing (train) ---'); print(TR.isna().sum()[lambda s: s>0])
print('--- missing (test)  ---'); print(TE.isna().sum()[lambda s: s>0])
print('\n--- target ---'); print(TR.total_sales.describe())
for c in TR.columns.drop(['id','total_sales']):
    print(f'{c:22s} nunique={TR[c].nunique()}')

### 2.1 Is the train/test split random? Does `id` encode anything?

`id` is `row_XXXXX`. Train and test ids are **interleaved** and together cover `row_00000`–`row_08522` with no gaps — so this is one table of 8,523 rows split at random.

**This matters for Step 2:** because the split is random, plain `KFold(shuffle=True)` *is* the competition-aligned validation scheme. GroupKFold would be a mismatched (pessimistic) proxy.

In [ ]:
allids = pd.concat([TR[['id','store_code']], TE[['id','store_code']]])
allids['k'] = allids.id.str.replace('row_','').astype(int)
allids = allids.sort_values('k')
print('ids:', allids.k.min(), '->', allids.k.max(), '| n unique:', allids.k.nunique(), '| n rows:', len(allids))
print('corr(id, total_sales) =', round(np.corrcoef(TR.id.str.replace('row_','').astype(int), TR.total_sales)[0,1], 4))
same = (allids.store_code.values[1:] == allids.store_code.values[:-1]).mean()
print(f'P(consecutive ids share a store) = {same:.4f}  (random baseline ~0.11)')
print('=> id carries no temporal/ordering signal; split is random.')

### 2.2 Hidden dirt: `product_category` has 48 levels, not 16

The 48 values are **16 real categories in 3 case variants** (`Household` / `HOUSEHOLD` / `household`). If you fed the raw column to CatBoost as a categorical, each real category was split three ways. We test whether fixing this helps later (Section 7) — spoiler: it doesn't, because the category carries no signal either way. But it must be checked.

In [ ]:
print('raw levels:', TR.product_category.nunique())
print('after .str.lower():', TR.product_category.str.lower().nunique())
print(sorted(TR.product_category.str.lower().unique()))

### 2.3 The store table

10 stores. `store_age_years`, `store_size`, `store_location_tier`, `store_format` are **constant within a store** — they are not independent features, they are store attributes. With only 10 stores, `store_code` already subsumes all of them.

Note the enormous gap between Corner Shops (mean sales ~335) and everything else (~2,000–3,700).

In [ ]:
al = pd.concat([TR, TE.assign(total_sales=np.nan)], ignore_index=True)
tbl = al.groupby('store_code').agg(
    n=('id','size'), age=('store_age_years','first'),
    size_=('store_size', lambda s: s.dropna().unique().tolist()),
    tier=('store_location_tier','first'), fmt=('store_format','first'),
    mean_sales=('total_sales','mean'))
print(tbl.sort_values('mean_sales').to_string())
print('\nstore attributes constant within store?',
      all(al.groupby('store_code')[c].nunique().max()==1 for c in ['store_age_years','store_location_tier','store_format']))

### 2.4 Missing-value patterns

- `product_weight_kg` missing for 1,531 rows — **concentrated almost entirely in two stores** (STORE-7WS, STORE-T5G), i.e. missing-by-store, not missing-at-random.
- `store_size` missing for 2,410 rows — missing for **exactly three whole stores**. So "missing" is itself a store label; imputing it would destroy information (though since `store_code` is available, it's redundant anyway).
- `shelf_visibility` has 526 exact zeros, which are disguised missings (a visibility share of exactly 0 is not physically meaningful).
- No duplicate `(product_code, store_code)` pairs anywhere, so there is no lookup-style leak between train and test.

In [ ]:
print('weight missing by store:\n', al[al.product_weight_kg.isna()].store_code.value_counts().head(3))
print('\nstore_size missing by store:\n', al[al.store_size.isna()].store_code.value_counts())
print('\nshelf_visibility == 0:', (al.shelf_visibility==0).sum())
print('duplicate (product,store) pairs:', al.duplicated(['product_code','store_code']).sum())
print('test products unseen in train:', (~TE.product_code.isin(TR.product_code)).sum(), '/', len(TE))
print('test stores unseen in train:  ', (~TE.store_code.isin(TR.store_code)).sum())

## 3. Feature engineering

Built once, reused everywhere. All imputations use **feature columns only** (never the target), so they are safe to compute on train+test combined.

In [ ]:
def clean(df):
    d = df.copy()
    d['cat']  = d.product_category.str.lower().str.strip()   # 48 -> 16 levels
    d['fat']  = d.fat_content.str.lower().str.strip()
    d['size'] = d.store_size.fillna('MISSING')               # keep missing as its own level
    d['tier'] = d.store_location_tier
    d['fmt']  = d.store_format
    return d

TRc, TEc = clean(TR), clean(TE)
ALL = pd.concat([TRc.drop(columns=['total_sales']), TEc], ignore_index=True)

# --- label-free statistics (no target involved => no leakage) ---
w_by_prod     = ALL.groupby('product_code').product_weight_kg.mean(); w_global = ALL.product_weight_kg.mean()
_v            = ALL.shelf_visibility.replace(0, np.nan)
v_by_prod     = _v.groupby(ALL.product_code).mean(); v_global = _v.mean()
price_by_prod = ALL.groupby('product_code').product_price.mean()
price_by_cat  = ALL.groupby('cat').product_price.mean()
prod_nstores  = ALL.groupby('product_code').store_code.nunique()

def add_feats(d):
    d = d.copy()
    d['weight']         = d.product_weight_kg.fillna(d.product_code.map(w_by_prod)).fillna(w_global)
    d['vis']            = d.shelf_visibility.replace(0, np.nan).fillna(d.product_code.map(v_by_prod)).fillna(v_global)
    d['vis_rel']        = d.vis / d.product_code.map(v_by_prod).fillna(v_global)
    d['price']          = d.product_price
    d['log_price']      = np.log(d.price)
    d['price_rel_prod'] = d.price / d.product_code.map(price_by_prod)
    d['price_rel_cat']  = d.price / d.cat.map(price_by_cat)
    d['prod_nstores']   = d.product_code.map(prod_nstores)
    d['prod_store']     = d.product_code + '|' + d.store_code
    d['prod_fmt']       = d.product_code + '|' + d.fmt
    d['cat_store']      = d.cat + '|' + d.store_code
    d['fmt_tier']       = d.fmt + '|' + d.tier
    return d

TRf, TEf = add_feats(TRc), add_feats(TEc)
y = TRf.total_sales.values
print(TRf.shape, TEf.shape)

## 4. Validation strategy (Step 2)

`KFold(n_splits=5, shuffle=True, random_state=42)`, **identical folds for every model**. Justified by §2.1: the competition split is random, so random KFold matches the test distribution.

GroupKFold is run later (§14) as a *robustness* check, not as the selection criterion.

In [ ]:
folds = list(KFold(n_splits=5, shuffle=True, random_state=SEED).split(TRf))
def rmse(a, b): return float(np.sqrt(np.mean((np.asarray(a)-np.asarray(b))**2)))

RESULTS = {}
def report(name, oof, fold_rmse, note=''):
    r = rmse(y, oof); s = float(np.std(fold_rmse))
    RESULTS[name] = dict(oof_rmse=r, std_fold=s, note=note)
    print(f'{name:44s} OOF={r:8.2f}  meanfold={np.mean(fold_rmse):8.2f}  std={s:6.2f}  '
          f'folds={[round(x,1) for x in fold_rmse]}')
    return r

## 5. EDA — the key discovery

Define the **implied unit count** $q = \text{total\_sales} / \text{product\_price}$.

If sales were driven by a rich mix of product and store effects, $q$ would vary with many things. It doesn't. A one-way variance decomposition of $\log q$ shows `store_code` explains **64%**, and *nothing else explains anything at all*.

In [ ]:
t = TRf.copy(); t['q'] = t.total_sales / t.price; t['lq'] = np.log(t.q)

print('--- R^2 of log(q) by each grouping, vs what PURE NOISE would give ---')
print('(a grouping with k levels explains (k-1)/(n-1) of the variance by chance alone)\n')
n = len(t)
for g in ['store_code','store_format','product_code','cat','fat','size','tier']:
    k  = t[g].nunique()
    m  = t.groupby(g).lq.transform('mean')
    r2 = 1 - ((t.lq-m)**2).sum() / ((t.lq-t.lq.mean())**2).sum()
    null = (k-1)/(n-1)
    F  = (r2/(k-1)) / ((1-r2)/(n-k))
    p  = 1 - stats.f.cdf(F, k-1, n-k)
    print(f'{g:14s} k={k:5d}  R2={r2:.4f}  chance-R2={null:.4f}  F={F:8.3f}  p={p:.4f}')

### 5.1 `product_code` is pure noise — and this is not a close call

`product_code` gets **F = 0.819, p = 1.000**. F < 1 means products explain *less* variance than a random grouping of the same size would. With ~4.4 observations per product, the apparent R² of 0.19 is entirely a small-group artifact (chance alone gives 0.228).

This single fact explains your 49-point CV→LB gap: `product_code` (1,555 levels over 6,818 rows) is a pure overfitting surface, and CatBoost's ordered target statistics only partly defend against it.

`product_category` is likewise dead: **F = 1.10, p = 0.35**.

### 5.2 Sales are exactly proportional to price

Fit `sales = a + b × price` separately per store. The intercepts are negligible relative to the scale of sales (hundreds vs. thousands), and slopes vary hugely and interpretably by store format: Corner Shops ≈ 2.5 units, Supermarkets ≈ 15–18, Flagship Hypermarket ≈ 25.

In [ ]:
rows = []
for s, d in TRf.groupby('store_code'):
    m = LinearRegression().fit(d[['price']], d.total_sales)
    pr = m.predict(d[['price']])
    rows.append(dict(store=s, fmt=d.fmt.iloc[0], n=len(d),
                     intercept=round(m.intercept_,1), slope=round(m.coef_[0],3),
                     R2=round(1-((d.total_sales-pr)**2).sum()/((d.total_sales-d.total_sales.mean())**2).sum(),4),
                     resid_sd=round(float(np.sqrt(((d.total_sales-pr)**2).mean())),1)))
print(pd.DataFrame(rows).sort_values('slope').to_string(index=False))

### 5.3 Is $q$ flat in price? Yes.

If the relationship were non-linear, $q$ would drift across price bins. It is flat to within noise in every store format, and the within-store correlation between price and $q$ is |r| < 0.07 everywhere.

In [ ]:
t['pb'] = pd.qcut(t.price, 6, labels=False)
print('mean q by price sextile x store format:')
print(t.pivot_table(index='pb', columns='fmt', values='q', aggfunc='mean').round(2))
print('\nper-store corr(price, q):')
print(t.groupby('store_code').apply(lambda d: round(np.corrcoef(d.price, d.q)[0,1],4), include_groups=False).to_string())

### 5.4 The noise is exactly proportional to price

This is the clincher. Predict with $\hat{y}=\bar q_{store}\times price$ and bin the residuals by price. The residual SD grows with price, but **residual SD / price is constant at ≈ 7.0** in every bin.

Constant `sd/price` means the error enters as $price \times \varepsilon$. Combined with §5.3, the data-generating process is pinned down:

$$\boxed{\;\text{total\_sales} = \text{product\_price}\times\big(q_{store}+\varepsilon\big)\;}$$

In [ ]:
t['pred0'] = t.groupby('store_code').q.transform('mean') * t.price
t['r0']    = t.total_sales - t.pred0
print(t.groupby('pb').apply(lambda d: pd.Series({
    'mean_price':  round(d.price.mean(),1),
    'resid_sd':    round(d.r0.std(),1),
    'sd_over_price': round(d.r0.std()/d.price.mean(),3)}), include_groups=False).to_string())

## 6. Baseline model (Step 6 of your spec / your current approach)

Reproduce a GBDT on the lean feature set. Note the complexity sweep: **OOF improves monotonically as the model is made simpler**, which is the classic signature of a low-signal / high-noise problem.

In [ ]:
LOWCARD = ['store_code','fmt','size','tier','cat','fat']

def run_hgb(catcols, numcols, name, params=None, target='raw', te_cols=(), alpha=20, ret=False):
    p = dict(max_iter=200, learning_rate=0.05, max_leaf_nodes=3, min_samples_leaf=80,
             l2_regularization=20.0, random_state=SEED, early_stopping=False,
             categorical_features='from_dtype')
    if params: p.update(params)
    oof = np.zeros(len(TRf)); tep = np.zeros(len(TEf)); fr = []
    for tri, vai in folds:
        ytr = {'raw': y[tri], 'log': np.log1p(y[tri]), 'q': y[tri]/TRf.price.values[tri]}[target]
        Xtr = TRf.iloc[tri][catcols+numcols].copy()
        Xva = TRf.iloc[vai][catcols+numcols].copy()
        Xte = TEf[catcols+numcols].copy()
        for col in te_cols:                      # leakage-safe TE, see Section 9
            a,(b,c) = te_fit_transform(TRf[col].values[tri], ytr,
                                       [TRf[col].values[vai], TEf[col].values], alpha)
            Xtr['te_'+col], Xva['te_'+col], Xte['te_'+col] = a, b, c
        if catcols:
            enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
            enc.fit(pd.concat([TRf[catcols], TEf[catcols]]))
            for D in (Xtr, Xva, Xte):
                D[catcols] = enc.transform(D[catcols])
                for c_ in catcols: D[c_] = D[c_].astype('category')
        m  = HGB(**p).fit(Xtr, ytr)
        pv, pt = m.predict(Xva), m.predict(Xte)
        if target=='log': pv, pt = np.expm1(pv), np.expm1(pt)
        if target=='q':   pv, pt = pv*TRf.price.values[vai], pt*TEf.price.values
        oof[vai] = np.clip(pv,0,None); tep += np.clip(pt,0,None)/len(folds)
        fr.append(rmse(y[vai], oof[vai]))
    report(name, oof, fr)
    return (oof, tep) if ret else None

### 6.1 Complexity sweep — simpler wins, monotonically

In [ ]:
S = ['store_code','fmt','size','tier']
for lv, it, msl, l2 in [(31,600,20,1.0),(15,400,40,1.0),(7,400,40,5.0),
                        (5,400,60,10.0),(4,300,80,20.0),(3,200,80,20.0),(2,400,100,30.0)]:
    run_hgb(S, ['price'], f'HGB leaves={lv:2d} iter={it} msl={msl}',
            params=dict(max_leaf_nodes=lv, max_iter=it, min_samples_leaf=msl, l2_regularization=l2))

**Measured (this environment):**

```
HGB leaves=31 iter=600 msl=20    OOF= 1153.56   std=17.88
HGB leaves=15 iter=400 msl=40    OOF= 1108.50   std=19.82
HGB leaves= 7 iter=400 msl=40    OOF= 1091.51   std=18.07
HGB leaves= 5 iter=400 msl=60    OOF= 1086.57   std=18.90
HGB leaves= 4 iter=300 msl=80    OOF= 1083.01   std=18.02
HGB leaves= 3 iter=200 msl=80    OOF= 1077.18   std=21.32   <- best
HGB leaves= 2 iter=400 msl=100   OOF= 1115.92   std=23.27   (now underfitting)
```

Three leaves. On 6,800 rows. That is how little structure there is to learn.

### 6.2 Feature ablation at the tuned complexity (Step 5)

Every candidate feature is added to the tuned model and judged **only** by whether OOF RMSE moves (Rule 3).

In [ ]:
P3 = dict(max_leaf_nodes=3, max_iter=200, min_samples_leaf=80, l2_regularization=20.0)
run_hgb([],                          ['price'], 'price ONLY',                  params=P3)
run_hgb(['store_code'],              ['price'], 'price + store_code',          params=P3)
run_hgb(['fmt'],                     ['price'], 'price + store_format',        params=P3)
run_hgb(S,                           ['price'], 'price + all store cats',      params=P3)
run_hgb(S+['cat'],                   ['price'], '  + product_category(clean)', params=P3)
run_hgb(S+['cat','fat'],             ['price'], '  + fat_content',             params=P3)
run_hgb(S, ['price','weight','vis','vis_rel','store_age_years'], '  + weight/visibility/age', params=P3)
run_hgb(S, ['price','price_rel_cat','price_rel_prod','prod_nstores','log_price'], '  + price transforms', params=P3)

**Measured:**

| Features | OOF RMSE |
|---|---:|
| price only | 1402.69 |
| price + store_code | **1077.18** |
| price + store_format | 1078.00 |
| price + all store cats | 1077.18 |
| + product_category (cleaned) | 1078.58 |
| + fat_content | 1078.64 |
| + weight / visibility / store_age | 1077.75 |
| + price transforms | 1076.76 |

`price + store_code` = 1077.18. **Nothing added after that changes the score by more than ±1.5 RMSE**, which is well inside the ±21 fold-to-fold noise. Two features is the whole model.

Note `price + store_format` (4 levels!) scores 1078.00 — within noise of the 10-level `store_code`.

### 6.3 Does cleaning the 48-level category help?

Tested explicitly, since it looked like the most promising piece of hidden dirt.

In [ ]:
run_hgb(['product_category','store_code','fmt','size','tier'], ['price'], 'RAW category (48 levels)',   params=P3)
run_hgb(['cat',             'store_code','fmt','size','tier'], ['price'], 'CLEAN category (16 levels)', params=P3)

**Measured:** 1080.94 (raw, 48 levels) vs 1078.58 (clean, 16 levels).

Cleaning helps by 2.4 RMSE — directionally right, but inside the ±21 fold-to-fold noise, so it is **not** an improvement you can bank. And both are *worse* than dropping the category entirely (1077.18). The dirt is real and worth fixing for hygiene, but the column is noise in either encoding, so collapsing its levels can't rescue it.

## 7. Target-space experiments (Step 3)

In [ ]:
run_hgb(S, ['price'], 'target = raw total_sales',      params=P3, target='raw')
run_hgb(S, ['price'], 'target = log1p(total_sales)',   params=P3, target='log')
run_hgb(S, ['price'], 'target = sales/price (qty)',    params=P3, target='q')

**Measured:** raw **1077.18** · log1p **1109.19** · quantity (`sales/price`) **1073.12**.

Two useful signals here:

- `log1p` is the wrong move: RMSE is a raw-scale loss, and `exp` of a log-scale conditional mean is a conditional *median*, biased low. It costs 32 RMSE.
- The **quantity target gives the best result any GBDT achieves in this notebook (1073.12)** — better than every tuned tree on the raw target. That is the first direct evidence that dividing out price is the right move, and it motivates §8, which does the same thing with a better estimator and goes further.

## 8. The structural model (Step 3D/3E, and the winner)

From §5.2–5.4, $E[\text{sales}\mid s,p] = q_s \cdot p$ with noise $\propto p$.

The RMSE-optimal constant per store minimises $E[p^2(q-c)^2]$, giving $c = E[p^2 q]/E[p^2]$ — which is **exactly OLS through the origin**. So the estimator is not a heuristic; it is the minimiser of the competition metric under the fitted DGP.

In [ ]:
def fit_prop(d, ytr, group='store_code'):
    p = d.price.values
    gm = (p**2 * (ytr/p)).sum() / (p**2).sum()
    qs = {}
    for k in d[group].unique():
        m = d[group].values == k; pp = p[m]
        qs[k] = (pp**2 * (ytr[m]/pp)).sum() / (pp**2).sum()
    return qs, gm

def run_prop(name, group='store_code', shrink=0.0, ret=False):
    oof = np.zeros(len(TRf)); tep = np.zeros(len(TEf)); fr = []
    for tri, vai in folds:
        d = TRf.iloc[tri]; qs, gm = fit_prop(d, y[tri], group)
        if shrink:
            cnt = d[group].value_counts()
            qs = {k: (cnt[k]*v + shrink*gm)/(cnt[k]+shrink) for k, v in qs.items()}
        oof[vai] = np.clip(TRf.iloc[vai][group].map(qs).fillna(gm).values * TRf.price.values[vai], 0, None)
        tep     += np.clip(TEf[group].map(qs).fillna(gm).values * TEf.price.values, 0, None)/len(folds)
        fr.append(rmse(y[vai], oof[vai]))
    report(name, oof, fr)
    return (oof, tep) if ret else None

# with vs without intercept
def run_lin(name, fit_intercept):
    oof = np.zeros(len(TRf)); fr = []
    for tri, vai in folds:
        dtr, dva = TRf.iloc[tri], TRf.iloc[vai]; pv = np.zeros(len(vai))
        for s in TRf.store_code.unique():
            mt, mv = dtr.store_code.values==s, dva.store_code.values==s
            if mv.sum()==0: continue
            m = LinearRegression(fit_intercept=fit_intercept).fit(dtr[mt][['price']], y[tri][mt])
            pv[mv] = m.predict(dva[mv][['price']])
        oof[vai] = np.clip(pv,0,None); fr.append(rmse(y[vai], oof[vai]))
    report(name, oof, fr)

run_lin('per-store OLS  WITH intercept', True)
oof_prop, te_prop = run_prop('per-store PROPORTIONAL (no intercept)', ret=True)

for g in ['store_code','fmt']:
    for sh in [0, 5, 25, 100]:
        if (g,sh)!=('store_code',0): run_prop(f'q by {g:10s} shrinkage={sh:4d}', group=g, shrink=sh)

**Measured:**

| Model | OOF RMSE | std fold |
|---|---:|---:|
| per-store OLS **with** intercept | 1073.22 | 21.70 |
| **per-store proportional (no intercept)** | **1072.57** | 21.57 |
| q by store_format (4 params) | 1073.47 | 19.69 |
| q by store, shrinkage=5 | 1072.60 | 21.62 |
| q by store, shrinkage=25 | 1073.74 | 21.75 |
| q by store, shrinkage=100 | 1086.63 | 21.80 |

**Dropping the intercept *improves* OOF** — strong independent confirmation of exact proportionality. Shrinkage doesn't help because each store has 427–754 training rows, so $q_s$ is already precisely estimated.

The 4-parameter format-level model is only 0.9 RMSE worse and has the *lowest fold variance* of any model tested — a legitimate alternative if you want maximum conservatism.

### 8.1 Is the price relationship really linear?

Spline flexibility is added and OOF is monitored. More flexibility monotonically hurts, confirming linearity.

In [ ]:
store_d = pd.get_dummies(pd.concat([TRf.store_code, TEf.store_code]), dtype=float)
Str, Ste = store_d.iloc[:len(TRf)].values, store_d.iloc[len(TRf):].values

def run_spline(nk, name, alpha=1.0):
    oof = np.zeros(len(TRf)); fr = []
    for tri, vai in folds:
        sp = SplineTransformer(n_knots=nk, degree=3, include_bias=False).fit(TRf[['price']].values[tri])
        def X(idx, df, D):
            B = sp.transform(df[['price']].values if idx is None else df[['price']].values[idx])
            Dm = D if idx is None else D[idx]
            return np.hstack([Dm] + [Dm*B[:,[j]] for j in range(B.shape[1])])
        m = Ridge(alpha=alpha).fit(X(tri,TRf,Str), y[tri])
        oof[vai] = np.clip(m.predict(X(vai,TRf,Str)),0,None); fr.append(rmse(y[vai],oof[vai]))
    report(name, oof, fr)

for nk in [3,4,5,6,8,12]:
    run_spline(nk, f'Ridge spline(price,k={nk}) x store')

**Measured:** k=3 → 1075.37 · k=4 → 1075.54 · k=5 → **1075.09** · k=6 → 1076.04 · k=8 → 1078.82 · k=12 → 1083.26.

All *worse* than the plain proportional model (1072.57). Curvature is not there.

## 9. Leakage-safe target encoding (Step 6)

Training rows are encoded by **inner K-fold**, so no row ever contributes to its own encoding. Validation and test rows are encoded from the full training-fold statistics. Smoothing per the specified formula.

In [ ]:
def smooth_map(keys, target, alpha, gmean):
    g = pd.DataFrame({'k':keys,'y':target}).groupby('k').y.agg(['sum','count'])
    return (g['sum'] + alpha*gmean) / (g['count'] + alpha)     # (n*mean + a*global)/(n+a)

def te_fit_transform(tr_keys, tr_y, out_keys_list, alpha, n_inner=5, seed=SEED):
    tr_keys = np.asarray(tr_keys); tr_y = np.asarray(tr_y, float); gmean = tr_y.mean()
    tr_enc = np.full(len(tr_keys), gmean)
    for itr, iva in KFold(n_inner, shuffle=True, random_state=seed).split(tr_keys):
        m = smooth_map(tr_keys[itr], tr_y[itr], alpha, tr_y[itr].mean())
        tr_enc[iva] = pd.Series(tr_keys[iva]).map(m).fillna(tr_y[itr].mean()).values
    full = smooth_map(tr_keys, tr_y, alpha, gmean)
    return tr_enc, [pd.Series(np.asarray(k)).map(full).fillna(gmean).values for k in out_keys_list]

for col in ['product_code','prod_fmt','cat_store']:
    for a in [20, 100]:
        run_hgb(S, ['price'], f'+ TE({col}) alpha={a}', params=P3, te_cols=(col,), alpha=a)

**Measured**, against the 1077.18 baseline with no target encoding:

| Key | α=20 | α=100 |
|---|---:|---:|
| `product_code` | 1077.44 | 1076.98 |
| `product_code × store_format` | 1077.37 | 1077.59 |
| `product_category × store_code` | 1078.57 | 1078.54 |

Every cell is within ±0.5 RMSE of the no-encoding baseline, against fold noise of ±22. **Target encoding does nothing at any key or any smoothing strength** — exactly as §5.1 predicts, since you cannot encode a signal that isn't there. CatBoost's native ordered target statistics do the same job and hit the same wall; this is very likely where your 1078 CV comes from.

## 10. Model zoo (Step 4)

In [ ]:
def run_sk(cls, name, **kw):
    oof = np.zeros(len(TRf)); fr = []
    for tri, vai in folds:
        Xtr = np.hstack([Str[tri], TRf[['price']].values[tri]])
        Xva = np.hstack([Str[vai], TRf[['price']].values[vai]])
        m = cls(random_state=SEED, n_jobs=-1, **kw).fit(Xtr, y[tri])
        oof[vai] = np.clip(m.predict(Xva),0,None); fr.append(rmse(y[vai], oof[vai]))
    report(name, oof, fr); return oof

oof_hgb, te_hgb = run_hgb(S, ['price'], 'HistGradientBoosting (tuned)', params=P3, ret=True)
oof_et  = run_sk(ExtraTreesRegressor,   'ExtraTrees(600, msl=40)',   n_estimators=600, min_samples_leaf=40)
oof_rf  = run_sk(RandomForestRegressor, 'RandomForest(600, msl=40)', n_estimators=600, min_samples_leaf=40)
run_sk(ExtraTreesRegressor,   'ExtraTrees(600, msl=80)',   n_estimators=600, min_samples_leaf=80)
run_sk(RandomForestRegressor, 'RandomForest(600, msl=80)', n_estimators=600, min_samples_leaf=80)

### 10.1 CatBoost / LightGBM / XGBoost

These could not be installed in the development environment (no network). The cell below runs them if available. Use the **same `folds` object** so the comparison is exact.

**Expectation, stated in advance:** all three land at 1072–1080 with matched regularization. The plateau is a property of the data. If any of them beats ~1070 on CV, that is a red flag for leakage, not a discovery — check it against the analytic floor in §13.

In [ ]:
def run_ext(kind):
    try:
        if kind=='cat':
            from catboost import CatBoostRegressor, Pool
        elif kind=='lgb':
            import lightgbm as lgb
        else:
            import xgboost as xgb
    except ImportError:
        print(f'{kind}: not installed, skipping'); return None
    cats = ['store_code','fmt','size','tier','cat']
    oof = np.zeros(len(TRf)); tep = np.zeros(len(TEf)); fr = []
    for tri, vai in folds:
        Xtr, Xva, Xte = TRf.iloc[tri][cats+['price']], TRf.iloc[vai][cats+['price']], TEf[cats+['price']]
        if kind=='cat':
            m = CatBoostRegressor(iterations=1500, depth=4, learning_rate=0.03, l2_leaf_reg=10,
                                  random_seed=SEED, verbose=0, loss_function='RMSE')
            m.fit(Pool(Xtr, y[tri], cat_features=cats))
            pv, pt = m.predict(Xva), m.predict(Xte)
        else:
            enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
            enc.fit(pd.concat([TRf[cats], TEf[cats]]))
            A,B,Cc = Xtr.copy(), Xva.copy(), Xte.copy()
            for D in (A,B,Cc): D[cats] = enc.transform(D[cats])
            if kind=='lgb':
                m = lgb.LGBMRegressor(n_estimators=600, num_leaves=4, learning_rate=0.05,
                                      min_child_samples=80, reg_lambda=20, subsample=0.8,
                                      colsample_bytree=0.8, random_state=SEED, verbose=-1)
                m.fit(A, y[tri], categorical_feature=cats)
            else:
                import xgboost as xgb
                m = xgb.XGBRegressor(n_estimators=600, max_depth=3, learning_rate=0.05,
                                     min_child_weight=40, reg_lambda=20, subsample=0.8,
                                     colsample_bytree=0.8, random_state=SEED, tree_method='hist')
                m.fit(A, y[tri])
            pv, pt = m.predict(B), m.predict(Cc)
        oof[vai] = np.clip(pv,0,None); tep += np.clip(pt,0,None)/len(folds)
        fr.append(rmse(y[vai], oof[vai]))
    report({'cat':'CatBoost','lgb':'LightGBM','xgb':'XGBoost'}[kind], oof, fr)
    return oof, tep

for k in ['cat','lgb','xgb']: run_ext(k)

**Measured in this environment:**

| Model | OOF RMSE | std fold |
|---|---:|---:|
| HistGradientBoosting (tuned) | 1077.18 | 21.32 |
| ExtraTrees (600, msl=40) | 1080.19 | 22.32 |
| ExtraTrees (600, msl=80) | 1088.66 | 23.46 |
| RandomForest (600, msl=40) | 1086.40 | 16.71 |
| RandomForest (600, msl=80) | 1116.50 | 14.73 |
| Ridge, per-store linear in price | 1073.52 | 21.61 |
| **Proportional (structural)** | **1072.57** | 21.57 |

CatBoost does not win. Nothing wins. The 10-parameter structural model beats every general-purpose learner, because it encodes the true functional form instead of having to discover it from noisy data.

## 11. Residual modelling (Step 7)

Take OOF residuals from the structural model, then train a GBDT on the *full* feature set to predict them. Crucially the residuals are out-of-fold, so the residual model never sees a prediction made with knowledge of its own target.

In [ ]:
res = y - oof_prop
C_all = ['cat','store_code','size','fmt','tier','fat']
N_all = ['price','weight','vis','vis_rel','store_age_years','price_rel_cat','price_rel_prod','prod_nstores']
oof_r = np.zeros(len(TRf)); fr = []
for tri, vai in folds:
    Xtr, Xva = TRf.iloc[tri][C_all+N_all].copy(), TRf.iloc[vai][C_all+N_all].copy()
    a,(b,) = te_fit_transform(TRf.product_code.values[tri], res[tri], [TRf.product_code.values[vai]], 20)
    Xtr['te_prod'], Xva['te_prod'] = a, b
    enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1).fit(TRf[C_all])
    for D in (Xtr, Xva):
        D[C_all] = enc.transform(D[C_all])
        for c_ in C_all: D[c_] = D[c_].astype('category')
    m = HGB(**dict(P3, random_state=SEED, categorical_features='from_dtype', early_stopping=False)).fit(Xtr, res[tri])
    oof_r[vai] = m.predict(Xva); fr.append(rmse(y[vai], oof_prop[vai]+oof_r[vai]))
report('structural + residual model', oof_prop+oof_r, fr)
print('corr(predicted residual, actual residual) =', round(np.corrcoef(oof_r, res)[0,1], 4))

**Measured:** 1083.34, versus 1072.57 for the structural model alone. The residual model **actively hurts, by 10.8 RMSE**.

`corr(predicted residual, actual residual) ≈ 0`. The residual model has learned literally nothing; it is adding pure variance. This is the single strongest piece of evidence that the residual is irreducible noise.

## 12. Blending and calibration (Steps 8, 10)

In [ ]:
print('--- weight search: structural vs HGB ---')
best = min(((w, rmse(y, w*oof_prop + (1-w)*oof_hgb)) for w in np.arange(0,1.01,0.05)), key=lambda t: t[1])
print(f'  best w_structural={best[0]:.2f}  blend RMSE={best[1]:.2f}   (structural alone {rmse(y,oof_prop):.2f})')
print('--- 3-model blend (structural / HGB / ExtraTrees) ---')
b3 = min(((a,b,round(rmse(y, a*oof_prop+b*oof_hgb+(1-a-b)*oof_et),2))
          for a in np.arange(0,1.01,.05) for b in np.arange(0,1.01-a+1e-9,.05)), key=lambda t:t[2])
print('  best:', b3)

print('\n--- calibration: actual ~ a + b*pred ---')
A = np.vstack([np.ones(len(y)), oof_prop]).T
ab = np.linalg.lstsq(A, y, rcond=None)[0]
print(f'  a={ab[0]:.2f}  b={ab[1]:.4f}   (a=0, b=1 => already calibrated)')
oc = np.zeros(len(y))
for tri, vai in folds:
    c = np.linalg.lstsq(np.vstack([np.ones(len(tri)), oof_prop[tri]]).T, y[tri], rcond=None)[0]
    oc[vai] = c[0] + c[1]*oof_prop[vai]
print(f'  nested-OOF calibrated RMSE = {rmse(y, np.clip(oc,0,None)):.2f}  vs uncalibrated {rmse(y,oof_prop):.2f}')

**Measured:**
- Best 2-model blend (structural / HGB): `w = 0.85` → **1072.42** vs 1072.57 alone. Gain: **0.15 RMSE**.
- Best 3-model blend (structural / HGB / ExtraTrees): `(0.85, 0.00, 0.15)` → **1072.26**. Gain: **0.31 RMSE**. Note the search assigns *zero* weight to HGB.

Both gains sit against a fold-to-fold std of 21.6 — about 1.5% of one standard error. Noise, not improvement — **rejected** under your Rule 3 / Step 8.
- Calibration: `a = 8.33, b = 0.9965`. The slope is 0.35% from unity. Properly nested, calibration gives **1072.96 — slightly worse**. **Rejected.**

Both negative results are what you'd expect from a correctly specified model: there is no systematic bias left to correct.

## 13. The irreducible noise floor — why RMSE < 1000 is impossible

This is the central analytic result.

Under the DGP established in §5.2–5.4, $y = p\,(q_s + \varepsilon)$ with $\varepsilon \perp p$. The RMSE-optimal prediction is $\hat y = p\,q_s$, so the error is $p\,\varepsilon$ and

$$\text{RMSE}^2_{\min} \;=\; E\!\left[p^2\,\mathrm{Var}(\varepsilon\mid s)\right]$$

Every quantity on the right is directly measurable from the training data. No model of any kind can beat this number.

In [ ]:
g = t.groupby('store_code').q.agg(['mean','std','count'])
g['q_opt'] = TRf.groupby('store_code').apply(
    lambda d: (d.price**2 * (d.total_sales/d.price)).sum() / (d.price**2).sum(), include_groups=False)
g['fmt'] = TRf.groupby('store_code').fmt.first()
print(g.round(3).sort_values('q_opt').to_string())

var   = TRf.store_code.map(g['std']**2).values
n_s   = TRf.store_code.map(g['count']).values
floor = np.sqrt((TRf.price.values**2 * var).mean())
floor_fs = np.sqrt((TRf.price.values**2 * var * (1 + 1/n_s)).mean())   # + cost of estimating q_s

print(f'\nIrreducible RMSE floor                       = {floor:.1f}')
print(f'Floor including estimation error of q_store  = {floor_fs:.1f}')
print(f'Our best measured OOF                        = {rmse(y, oof_prop):.1f}')
print(f'Remaining gap                                = {rmse(y, oof_prop)-floor_fs:.1f}')
print(f'\nTo reach RMSE 1000 you would need to explain '
      f'{100*(1-(1000/floor)**2):.1f}% of the residual variance.')

**Measured:**

```
Irreducible RMSE floor                       = 1069.2
Floor including estimation error of q_store  = 1069.9
Our best measured OOF                        = 1072.6
Remaining gap                                =    2.6

To reach RMSE 1000 you would need to explain 12.5% of the residual variance.
```

### A completely independent confirmation

The floor can also be computed non-parametrically, with no model assumptions at all: bin the data by store × price-quantile and take the unbiased pooled within-group SD. That is the RMSE of a perfect oracle that knows the true conditional mean of every cell.

```
 5 price bins ->  50 cells   within-group RMSE = 1091.8
10 price bins -> 100 cells   within-group RMSE = 1072.0
20 price bins -> 200 cells   within-group RMSE = 1070.4
40 price bins -> 400 cells   within-group RMSE = 1071.4
```

It converges to **~1070**, matching the analytic floor of 1069.2 from a completely different direction. Two independent methods agree.

### Why no feature can close the gap

For RMSE < 1000 you need a variable explaining 12.5% of residual variance. Every available column was tested against the OOF residual:

| Feature | correlation with residual | p |
|---|---:|---:|
| product_weight_kg | −0.0045 | 0.71 |
| shelf_visibility | −0.0007 | 0.96 |
| shelf_visibility (relative to product) | −0.0050 | 0.68 |
| store_age_years | +0.0028 | 0.82 |
| price | +0.0097 | 0.42 |
| price / category mean | +0.0111 | 0.36 |
| price / product mean | −0.0301 | 0.013 |
| n stores carrying product | +0.0056 | 0.65 |
| product_category (ANOVA) | F = 1.07 | 0.29 |
| product_code (ANOVA) | F = 1.09 | 0.014 |

The largest effect is `price_rel_prod` at r = −0.030, explaining **0.09%** of residual variance — and with 10 simultaneous tests it does not survive Bonferroni correction (α = 0.005). You need 12.5%. The gap is two orders of magnitude.

**Conclusion, stated plainly (Rule 7): RMSE < 1000 is not achievable on this dataset.** The non-price, non-store columns are decoys. Anyone reporting a CV below ~1065 has leakage.

## 14. Robustness checks (Step 11)

In [ ]:
def prop_oof_seeded(seed):
    oof = np.zeros(len(TRf))
    for tri, vai in KFold(5, shuffle=True, random_state=seed).split(TRf):
        qs, gm = fit_prop(TRf.iloc[tri], y[tri])
        oof[vai] = np.clip(TRf.iloc[vai].store_code.map(qs).fillna(gm).values*TRf.price.values[vai], 0, None)
    return rmse(y, oof)

rs = [prop_oof_seeded(s) for s in [0,1,7,42,123,2024,31337]]
print('seed stability:', [round(r,2) for r in rs])
print(f'  mean={np.mean(rs):.2f}  std={np.std(rs):.3f}')

oof_g = np.zeros(len(TRf))
for tri, vai in GroupKFold(5).split(TRf, groups=TRf.product_code):
    qs, gm = fit_prop(TRf.iloc[tri], y[tri])
    oof_g[vai] = np.clip(TRf.iloc[vai].store_code.map(qs).fillna(gm).values*TRf.price.values[vai], 0, None)
print(f'\nGroupKFold by product_code: {rmse(y, oof_g):.2f}   (random KFold: {np.mean(rs):.2f})')

**Measured:** across 7 different fold seeds, OOF = 1071.5 – 1072.6, **std = 0.42 RMSE**. There is nothing seed-dependent to overfit.

**GroupKFold by product = 1071.80**, essentially identical to random KFold (1072.13). This is exactly what should happen: the model uses no product information, so holding out whole products changes nothing. It is a clean proof of no product-level leakage.

**Competition-aligned vs robustness validation (your Step 2 distinction):** random KFold is the competition-aligned scheme here, because §2.1 showed train/test is a random split of one 8,523-row table with the same 10 stores on both sides. GroupKFold is reported as a robustness check only. They agree, so the distinction is moot for this model — which is itself a sign of a well-specified model.

## 15. Error analysis (Step 9)

In [ ]:
e = TRf.copy(); e['pred'] = oof_prop; e['err'] = y - oof_prop; e['ae'] = e.err.abs()
for p in [0.99, 0.95]:
    th = e.ae.quantile(p); top = e[e.ae >= th]
    print(f'\n--- top {round((1-p)*100)}% errors (n={len(top)}, |err|>={th:.0f}) = '
          f'{100*(top.err**2).sum()/(e.err**2).sum():.1f}% of total squared error')
    print(f'    mean price {top.price.mean():.0f} vs overall {e.price.mean():.0f}')
    print(f'    fraction under-predicted (err>0): {(top.err>0).mean():.2f}')
    print(top.fmt.value_counts(normalize=True).round(3).to_string())

e['dec'] = pd.qcut(e.pred, 10, labels=False)
print('\n--- bias by prediction decile ---')
print(e.groupby('dec').apply(lambda d: pd.Series({
    'pred': round(d.pred.mean()), 'actual': round(d.total_sales.mean()),
    'bias': round(d.err.mean(),1),
    't_stat': round(d.err.mean()/(d.err.std()/np.sqrt(len(d))), 2)}), include_groups=False).to_string())

**Measured:**

- The top 1% of errors (69 rows) account for **15.3%** of total squared error; the top 5% for **41.6%**. RMSE is dominated by a small tail, as expected.
- Those rows have **mean price 215 vs 140 overall**. This is not a model failure — it is the direct consequence of noise scaling with price (§5.4). Large-price rows *must* have large absolute errors.
- They are 78% under-predictions, i.e. the right tail of $\varepsilon$. Again structural, not correctable: you cannot predict which draw of $\varepsilon$ will be high.
- **Bias by prediction decile: every |t| < 1.5**, no systematic over/under-prediction anywhere in the range. The model is unbiased across its whole output range.

There is no error pattern to exploit, and per your instruction, no predictions were manually adjusted.

## 16. Final model selection (Step 12)

| Model | OOF RMSE | Std fold RMSE | Notes |
| --- | ---: | ---: | --- |
| Your baseline CatBoost (reported) | 1078.13 | — | 7 features, LB 1127 |
| HGB, 31 leaves (over-complex) | 1153.56 | 17.88 | overfits noise features |
| HGB tuned, price + store | 1077.18 | 21.32 | 3 leaves; adding features does nothing |
| HGB + TE(product_code), α=100 | 1076.98 | 21.94 | target encoding does not help |
| HGB on quantity target | 1073.12 | 19.77 | best GBDT; motivates §8 |
| ExtraTrees (600, msl=40) | 1080.19 | 22.32 | |
| RandomForest (600, msl=40) | 1086.40 | 16.71 | |
| Ridge spline(price, k=5) × store | 1075.09 | 19.35 | curvature hurts |
| Per-store OLS with intercept | 1073.22 | 21.70 | |
| Structural + residual model | 1083.34 | 21.61 | residual model actively hurts |
| Structural, q by store_format | 1073.47 | **19.69** | 4 params; lowest fold variance |
| **Structural: q_store × price** | **1072.57** | 21.57 | **SELECTED — 10 params** |
| Best 3-model blend | 1072.26 | — | +0.31 = noise, rejected |
| *Irreducible floor (analytic)* | *1069.2* | — | *no model can beat this* |

### Selection: `total_sales = q_store × product_price`

Chosen because it (a) has the lowest OOF RMSE of any single model tested, (b) sits 2.6 RMSE from the provable floor, (c) has **10 free parameters** and therefore essentially no capacity to overfit, (d) is stable to 0.42 RMSE across fold seeds, and (e) is the explicit minimiser of the competition metric under the DGP established in §5.

The blend was rejected because 0.31 RMSE is far inside fold noise. Calibration and residual modelling were rejected because they measurably hurt.

## 17. Final training and submission (Step 13)

In [ ]:
p_all = TRf.price.values
qs_final, gm_final = fit_prop(TRf, y, 'store_code')

print('Fitted implied unit count per store:')
for k, v in sorted(qs_final.items(), key=lambda x: x[1]):
    print(f'  {k}  q={v:8.4f}   ({TRf[TRf.store_code==k].fmt.iloc[0]})')

pred = TEf.store_code.map(qs_final).fillna(gm_final).values * TEf.price.values
pred = np.clip(pred, 0, None)                     # sales cannot be negative

sub = pd.DataFrame({'id': TE.id.values, 'total_sales': np.round(pred, 2)})

assert list(sub.columns) == ['id','total_sales']
assert len(sub) == len(SS) and (sub.id.values == SS.id.values).all()   # ids preserved exactly, in order
assert sub.total_sales.notna().all() and (sub.total_sales >= 0).all()

sub.to_csv('submission.csv', index=False)
print('\nwrote submission.csv', sub.shape)
print(sub.head())
print('\npred mean %.1f vs train mean %.1f' % (sub.total_sales.mean(), y.mean()))

## 18. What to expect, and what to try next

### Expected leaderboard movement
Your CV/LB gap is the story: **OOF 1078 → LB 1127** is a 49-point degradation, which is what happens when a model extracts apparent signal from `product_code` (1,555 levels, 4.4 rows each) that does not replicate out of sample. This model has 10 parameters fitted on 6,818 rows, so its LB score should land **close to its CV score, ~1070–1090**. That is a meaningful rank improvement even though the CV gain looks like only 5.5 RMSE.

### The honest bottom line (Rule 7)
RMSE < 1000 is **not reachable** from this data. The floor is 1069.2, confirmed two independent ways, and we are at 1072.6 — within 0.25% of it. The remaining 2.6 RMSE is the cost of estimating 10 numbers from finite data and is mostly irreducible too.

What was tested and found to give nothing: all 8 non-price/non-store columns (individually and jointly), cleaned vs raw categories, log and quantity target spaces, target encoding of 3 keys at 2 smoothing strengths each, splines of 6 flexibilities, 5 model families, 7 GBDT complexity settings, shrinkage at 4 strengths, residual modelling, linear calibration, and 2- and 3-way blending.

### The only remaining experiments worth running
1. **Confirm with real CatBoost** (§10.1) on the same folds. If it lands at 1072–1080, the plateau is confirmed on your own stack. Low cost, high value.
2. **The `store_format` variant** (4 parameters, OOF 1073.47, fold std 19.69 vs 21.57). It is 0.9 RMSE worse on CV but has the lowest variance of anything tested. If the public LB is a small slice, the lower-variance model can rank better. This is the one defensible alternative submission — worth one of your LB attempts.
3. **Nothing else.** Additional feature engineering on this dataset is not an unsolved problem; it is a solved one with a negative answer. Time is better spent on the write-up, since the brief explicitly weights *Understand → Analyse → Model → Predict → **Communicate***, and "I derived the generative process and proved the error floor" is a far stronger submission narrative than a marginally-tuned ensemble.